# Ask CIP v2: more ways to say every question, a reader taught on lakhs of them, tested on lakhs more

**Runtime → Change runtime type → A100 GPU** (L4 also works, slower), High-RAM on. Then **Runtime → Run all**.

* The first cell asks you to choose **cip-colab-v2-inputs.zip**: upload it from your computer (Downloads/Collab).
* About 3 hours on an A100 (about 35-45 compute units). No paid AI is used anywhere: the questions are written by an open model (Qwen3) on this GPU.
* Keep the tab open: if Colab disconnects, Run all again in the same session and the stages already done are skipped.
* At the end **cip-v2-results.zip** downloads by itself. Send it to Claude: nothing is shipped until it beats today's Ask on every test set.

In [ ]:
import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
from google.colab import files

SAVE = Path('/content/cip-v2'); SAVE.mkdir(parents=True, exist_ok=True)   # every stage's output: a rerun skips what is done
WORK = Path('/content/cip')
if not (WORK / 'scripts').exists():
    print('Choose cip-colab-v2-inputs.zip')
    uploaded = files.upload()
    name = next((n for n in uploaded if n.startswith('cip-colab-v2-inputs')), None)
    if name is None:
        raise ValueError('Choose the file named cip-colab-v2-inputs.zip and run this cell again.')
    with zipfile.ZipFile(Path('/content') / name) as z: z.extractall(WORK)
    del uploaded
    # the training copy of the publication shares the fresh one's question files (it only stops at an earlier wave)
    link = WORK / 'data/train/api/v1/questions'
    if not link.exists(): link.symlink_to(WORK / 'data/fresh/api/v1/questions')
os.chdir(WORK)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout,
      os.cpu_count(), 'CPUs')

def run(*args, **env):
    """One stage as its own process (a GPU stage frees the GPU when it ends); stops the notebook on failure."""
    started = time.time()
    done = subprocess.run([sys.executable, *map(str, args)], env={**os.environ, 'PYTHON_DOTENV_DISABLED': '1', **env})
    print(f'— {args[1]} {"done" if done.returncode == 0 else "FAILED"} in {(time.time() - started) / 60:.1f} min', flush=True)
    if done.returncode: raise SystemExit(done.returncode)

In [ ]:
# libraries: vLLM (the question writer, it brings torch), the trainer's export, Ask's own few
!pip -q install vllm
!pip -q install transformers onnx onnxscript onnxruntime tokenizers sqlalchemy pyyaml
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

## 1. Write 64 analyst-style questions for each of the 2,071 survey question wordings (GPU, ~30-60 min)

In [ ]:
V2 = 'scripts/colab/ask_cip_v2.py'
run(V2, 'expand', 'data/wordings.jsonl.gz', '--out', SAVE / 'expanded.jsonl', '--per', 64)

## 2. Half become Ask's extra search wordings, half a question bank whose right answer is known

In [ ]:
if not (SAVE / 'bank.csv.gz').exists():
    run(V2, 'split', '--wordings', 'data/wordings.jsonl.gz', '--expanded', SAVE / 'expanded.jsonl', '--semantic', 'config/ask_semantic.json',
        '--semantic-out', SAVE / 'ask_semantic.json', '--bank-out', SAVE / 'bank.csv.gz')
NEW = {'CIP_ASK_SEMANTIC': str(SAVE / 'ask_semantic.json')}
run(V2, 'embed', '--published', 'data/fresh/api/v1', **NEW)

## 3. Training cases: every bank question through Ask's search, in parallel (CPU)

In [ ]:
CASES = SAVE / 'cases'
if not (CASES / 'cases.jsonl').exists():
    run(V2, 'shards', '--bank', SAVE / 'bank.csv.gz', '--train-published', 'data/train/api/v1', '--test-published', 'data/fresh/api/v1',
        '--output', CASES, '--expert', 'data/ask-reviewed.jsonl', **NEW)
    run(V2, 'teach', '--cases', CASES / 'cases.jsonl', '--baseline', 'models/ask-reader', **NEW)
run('scripts/colab/ask_cip_train.py', 'preflight', '--cases', CASES / 'cases.jsonl', '--baseline', 'models/ask-reader',
    '--output', SAVE / 'preflight-report.json')

## 4. Train the reader on the GPU, today's reader as its teacher (~20-40 min)

In [ ]:
CAND = SAVE / 'candidate'
if not (CAND / 'paired-report.json').exists():
    shutil.rmtree(CAND, ignore_errors=True)
    run('scripts/colab/ask_cip_train.py', 'train', '--cases', CASES / 'cases.jsonl', '--baseline', 'models/ask-reader', '--output', CAND,
        '--epochs', 2, '--batch', 16, '--lr', 3e-5, '--teacher-weight', 0.5)

## 5. Test at scale: lakhs of questions through the whole of Ask, three ways (CPU, ~1 hour)

In [ ]:
shutil.copy('docs/research/ask-question-bank.csv.gz', SAVE / 'old-bank.csv.gz')
tests = [('today', 'models/ask-reader', 'config/ask_semantic.json'),
         ('new-wordings', 'models/ask-reader', SAVE / 'ask_semantic.json'),
         ('new-wordings-new-reader', CAND, SAVE / 'ask_semantic.json')]
for name, reader, semantic in tests:
    for bank in ('bank', 'old-bank'):
        report = SAVE / f'replay-{name}-{bank}.json'
        if not report.exists():
            run(V2, 'replay', '--bank', SAVE / f'{bank}.csv.gz', '--published', 'data/fresh/api/v1', '--reader', reader,
                '--semantic', semantic, '--report', report)

## 6. The verdict, and the results file for Claude

In [ ]:
rows = []
for name, _, _ in tests:
    for bank in ('bank', 'old-bank'):
        r = json.loads((SAVE / f'replay-{name}-{bank}.json').read_text())
        rows.append((name, bank, r['questions'], r['all']['share'], r['test']['share'], r['ask_golden.yml'][:22], r['ask_fresh.yml'][:20], r['ask_researched.yml'][:20]))
print(f"{'setup':26} {'bank':9} {'questions':>9} {'right':>6} {'unseen':>7}  golden / fresh / researched")
for row in rows: print(f'{row[0]:26} {row[1]:9} {row[2]:>9} {row[3]:>6.1%} {row[4]:>7.1%}  {row[5]} / {row[6]} / {row[7]}')
out = Path('/content/cip-v2-results.zip')
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in [SAVE / 'ask_semantic.json', SAVE / 'preflight-report.json', *SAVE.glob('replay-*.json'), *SAVE.glob('replay-*.misses.jsonl.gz'),
                 *(CAND.glob('*') if CAND.exists() else [])]:
        z.write(path, path.relative_to(SAVE))
files.download(str(out))